# AfriVoices ASR — Fine-tune Whisper-small (Google Colab)

**Before running:** add these 3 secrets via the 🔑 key icon in the left sidebar:
- `HF_TOKEN` — HuggingFace token with **Write** permission (huggingface.co → Settings → Tokens)
- `KAGGLE_USERNAME` — your Kaggle username (from kaggle.json)
- `KAGGLE_KEY` — your Kaggle API key (from kaggle.json)

To get `kaggle.json`: kaggle.com → Profile → Settings → API → Create New Token

In [ ]:
# ── CELL 1 — Mount Drive + Install ──────────────────────────────────────────
# Google Drive persists files across Colab disconnects — model saves here
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/afrivoices", exist_ok=True)

!pip install -q -U transformers accelerate datasets evaluate jiwer \
                  soundfile librosa pydub huggingface_hub kagglehub
print("Dependencies installed.")

In [ ]:
# ── CELL 2 — Secrets + Imports ──────────────────────────────────────────────
from google.colab import userdata
import os, io, glob, warnings
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
import evaluate
from huggingface_hub import login, HfApi

# Load secrets from Colab secret manager (🔑 icon in left sidebar)
HF_TOKEN        = userdata.get("HF_TOKEN")
KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY      = userdata.get("KAGGLE_KEY")

# Kaggle credentials must be env vars for kagglehub to pick them up
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

login(token=HF_TOKEN, quiet=True)
print("HuggingFace login OK.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL_ID       = "openai/whisper-small"
CHECKPOINT_DIR = "/content/drive/MyDrive/afrivoices/whisper-small-swa-som"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
# ── CELL 3 — Load Whisper processor ─────────────────────────────────────────
processor       = WhisperProcessor.from_pretrained(MODEL_ID)
feature_extractor = processor.feature_extractor
tokenizer       = processor.tokenizer
print(f"Processor loaded: {MODEL_ID}")

In [ ]:
# ── CELL 4 — Audio decode helper ────────────────────────────────────────────
def decode_audio(audio_field):
    """Accept HF Audio dict, bytes dict, or raw bytes → 16 kHz float32 array."""
    if isinstance(audio_field, dict) and "array" in audio_field:
        arr = np.array(audio_field["array"], dtype=np.float32)
        sr  = audio_field.get("sampling_rate", 16000)
        if sr != 16000:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        return arr

    raw = audio_field.get("bytes") if isinstance(audio_field, dict) else audio_field
    if isinstance(raw, bytes) and len(raw) > 0:
        try:
            arr, sr = sf.read(io.BytesIO(raw), dtype="float32")
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            if sr != 16000:
                import librosa
                arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
            return arr
        except Exception:
            pass
        from pydub import AudioSegment
        seg = (AudioSegment.from_file(io.BytesIO(raw))
               .set_frame_rate(16000).set_channels(1))
        return np.array(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

    raise ValueError(
        f"Cannot decode audio — got type {type(audio_field)}, "
        f"dict keys: {list(audio_field.keys()) if isinstance(audio_field, dict) else 'N/A'}"
    )

print("decode_audio helper ready.")

In [ ]:
# ── CELL 5 — Load Swahili training data ─────────────────────────────────────
import json, tarfile
from huggingface_hub import hf_hub_download

N_SAMPLES = 5000
SHARD     = 0
print(f"Loading {N_SAMPLES} Swahili clips from shard {SHARD}...")

manifest_path = hf_hub_download(
    repo_id="DigitalUmuganda/Afrivoice_Swahili",
    filename=f"agriculture_swahili_train/manifest_{SHARD}.jsonl",
    repo_type="dataset",
    token=HF_TOKEN,
)
with open(manifest_path) as f:
    all_entries = [json.loads(l) for l in f]

wanted = {}
for entry in all_entries:
    text = (entry.get("normalized_transcription") or "").strip()
    if text:
        wanted[entry["key"]] = text
    if len(wanted) >= N_SAMPLES:
        break
print(f"  {len(wanted)} valid manifest entries.")

audio_tar_path = hf_hub_download(
    repo_id="DigitalUmuganda/Afrivoice_Swahili",
    filename=f"agriculture_swahili_train/audio/audio_{SHARD}.tar.xz",
    repo_type="dataset",
    token=HF_TOKEN,
)
print(f"  Audio tar downloaded. Processing...")

swa_records = []
tokenizer.set_prefix_tokens(language="swahili", task="transcribe")

with tarfile.open(audio_tar_path, "r:xz") as tar:
    for member in tar:
        if not member.name.endswith(".webm"):
            continue
        key = os.path.basename(member.name).replace(".webm", "")
        if key not in wanted:
            continue
        try:
            arr = decode_audio({"bytes": tar.extractfile(member).read()})
            arr = arr[:480_000]
        except Exception as e:
            print(f"  SKIP {key}: {e}")
            continue
        input_features = feature_extractor(arr, sampling_rate=16000).input_features[0]
        labels = tokenizer(wanted[key]).input_ids
        swa_records.append({"input_features": input_features, "labels": labels})
        if len(swa_records) % 500 == 0:
            print(f"  {len(swa_records)} / {N_SAMPLES} clips", flush=True)
        if len(swa_records) >= N_SAMPLES:
            break

print(f"\nSwahili: {len(swa_records)} clips ready.")

In [ ]:
# ── CELL 6 — Load Somali training data (multi-shard) ────────────────────────
import lzma
from huggingface_hub import hf_hub_download

SOM_TARGET = 2000

# Step 1: scan manifest_{shard}.json files until we have enough valid entries
print("Scanning Somali manifests for valid transcriptions...")
wanted_by_shard = {}   # shard_id → {stem: text}
total_wanted    = 0

for shard in range(171):
    manifest_path = hf_hub_download(
        repo_id="DigitalUmuganda/Afrivoice",
        filename=f"Somali/manifest_{shard}.json",
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with open(manifest_path) as f:
        entries = [json.loads(line) for line in f if line.strip()]

    shard_wanted = {}
    for entry in entries:
        text = (entry.get("transcription") or
                entry.get("normalized_transcription") or "").strip()
        audio_fp = entry.get("audio_filepath", "")
        key = os.path.splitext(os.path.basename(str(audio_fp)))[0]
        if text and key:
            shard_wanted[key] = text

    if shard_wanted:
        wanted_by_shard[shard]  = shard_wanted
        total_wanted           += len(shard_wanted)
        print(f"  shard {shard}: {len(shard_wanted)} valid  (total: {total_wanted})",
              flush=True)

    if total_wanted >= SOM_TARGET:
        break

print(f"\nScanned {len(wanted_by_shard)} shards, {total_wanted} valid entries.")

# Step 2: download audio tars only for shards that have valid entries
som_records = []
tokenizer.set_prefix_tokens(language="somali", task="transcribe")

for shard, wanted in wanted_by_shard.items():
    if len(som_records) >= SOM_TARGET:
        break
    print(f"\nDownloading audio_{shard}.tar.xz ({len(wanted)} targets)...")

    for attempt in range(2):
        audio_tar_path = hf_hub_download(
            repo_id="DigitalUmuganda/Afrivoice",
            filename=f"Somali/audio_shards/audio_{shard}.tar.xz",
            repo_type="dataset",
            token=HF_TOKEN,
            force_download=(attempt > 0),
        )
        try:
            with tarfile.open(audio_tar_path, "r:xz") as tar:
                for member in tar:
                    if member.isdir():
                        continue
                    key = os.path.splitext(os.path.basename(member.name))[0]
                    if key not in wanted:
                        continue
                    try:
                        arr = decode_audio({"bytes": tar.extractfile(member).read()})
                        arr = arr[:480_000]
                    except Exception:
                        continue
                    input_features = feature_extractor(
                        arr, sampling_rate=16000).input_features[0]
                    labels = tokenizer(wanted[key]).input_ids
                    som_records.append({"input_features": input_features,
                                        "labels": labels})
                    if len(som_records) % 200 == 0:
                        print(f"  {len(som_records)} Somali clips", flush=True)
                    if len(som_records) >= SOM_TARGET:
                        break
            break  # success
        except (lzma.LZMAError, tarfile.TarError) as e:
            print(f"  Shard {shard} corrupt (attempt {attempt+1}): {e}")
            if attempt == 1:
                print(f"  Skipping shard {shard}.")

print(f"\nSomali: {len(som_records)} clips ready.")

In [ ]:
# ── CELL 7 — Build combined dataset + train/eval split ───────────────────────
from torch.utils.data import Dataset as TorchDataset

class WhisperDataset(TorchDataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        return self.records[i]

all_records = swa_records + som_records
np.random.shuffle(all_records)

split    = int(0.95 * len(all_records))
train_ds = WhisperDataset(all_records[:split])
eval_ds  = WhisperDataset(all_records[split:])
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

In [ ]:
# ── CELL 8 — Data collator + WER metric ──────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = torch.tensor(
            np.stack([f["input_features"] for f in features]), dtype=torch.float32
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        return {"input_features": input_features, "labels": labels}


wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": round(wer_metric.compute(
        predictions=pred_str, references=label_str), 4)}


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids(
        "<|startoftranscript|>"),
)
print("Data collator + WER metric ready.")

In [ ]:
# ── CELL 9 — Load model + training args ──────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids          = None
model.config.suppress_tokens             = []
model.generation_config.forced_decoder_ids = None
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-5,
    warmup_steps                = 100,
    max_steps                   = 600,   # ~3.5 hours on T4
    gradient_checkpointing      = True,
    fp16                        = True,
    eval_strategy               = "steps",
    per_device_eval_batch_size  = 8,
    predict_with_generate       = True,
    generation_max_length       = 225,
    save_steps                  = 200,
    eval_steps                  = 200,
    logging_steps               = 25,
    report_to                   = ["tensorboard"],
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    push_to_hub                 = False,
    dataloader_num_workers      = 2,
)
print("Training args configured.")

In [ ]:
# ── CELL 10 — Train ──────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = feature_extractor,
)

print("Starting fine-tuning...")
print(f"  {len(train_ds)} train  |  {len(eval_ds)} eval")
print(f"  max_steps=600, effective batch=32")
trainer.train()
print("Training done.")

# Save to Google Drive
trainer.save_model(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f"Model saved to Google Drive: {CHECKPOINT_DIR}")

# Push to HuggingFace Hub immediately as backup
api  = HfApi(token=HF_TOKEN)
user = api.whoami()
REPO_ID = f"{user['name']}/afrivoices-whisper-small-swa-som"
api.create_repo(REPO_ID, exist_ok=True, private=True)
model.push_to_hub(REPO_ID, token=HF_TOKEN)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f"Model pushed to HF Hub: https://huggingface.co/{REPO_ID}")

In [ ]:
# ── CELL 11 — Reload fine-tuned model for inference ──────────────────────────
import gc

# Free training objects before loading inference copy
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

ft_processor = WhisperProcessor.from_pretrained(CHECKPOINT_DIR)
ft_model     = WhisperForConditionalGeneration.from_pretrained(
    CHECKPOINT_DIR, torch_dtype=torch.float16
).to(device)
ft_model.eval()
ft_model.config.forced_decoder_ids            = None
ft_model.generation_config.forced_decoder_ids = None
ft_model.generation_config.suppress_tokens    = []
print(f"Fine-tuned model loaded from {CHECKPOINT_DIR}")

In [ ]:
# ── CELL 12 — Transcribe test set ────────────────────────────────────────────
# decode_audio is redefined here so this cell survives a kernel restart
import kagglehub, glob

def decode_audio(audio_field):
    if isinstance(audio_field, dict) and "array" in audio_field:
        arr = np.array(audio_field["array"], dtype=np.float32)
        sr  = audio_field.get("sampling_rate", 16000)
        if sr != 16000:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        return arr
    raw = audio_field.get("bytes") if isinstance(audio_field, dict) else audio_field
    if isinstance(raw, bytes) and len(raw) > 0:
        try:
            arr, sr = sf.read(io.BytesIO(raw), dtype="float32")
            if arr.ndim > 1:
                arr = arr.mean(axis=1)
            if sr != 16000:
                import librosa
                arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
            return arr
        except Exception:
            pass
        from pydub import AudioSegment
        seg = (AudioSegment.from_file(io.BytesIO(raw))
               .set_frame_rate(16000).set_channels(1))
        return np.array(seg.get_array_of_samples(), dtype=np.float32) / 32768.0
    raise ValueError(f"Cannot decode audio — type {type(audio_field)}")


def transcribe_batch(arrays, language=None):
    arrays = [a[:480_000] for a in arrays]
    inputs = ft_processor(
        arrays, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device).to(torch.float16)
    gen_kwargs = {"max_new_tokens": 225}
    if language:
        lang_id   = ft_processor.tokenizer.convert_tokens_to_ids(f"<|{language}|>")
        task_id   = ft_processor.tokenizer.convert_tokens_to_ids("<|transcribe|>")
        nots_id   = ft_processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")
        gen_kwargs["forced_decoder_ids"] = [[1, lang_id], [2, task_id], [3, nots_id]]
    with torch.no_grad():
        ids = ft_model.generate(input_features=inputs, **gen_kwargs)
    return ft_processor.batch_decode(ids, skip_special_tokens=True)


LANG_TO_WHISPER = {
    "swa": "sw",
    "som": "so",
    "kik": None,
    "luo": None,
    "mas": None,
    "kln": None,
}

test_path       = kagglehub.dataset_download("digitalumuganda/anv-test-data-nt")
CHECKPOINT_FILE = "/content/drive/MyDrive/afrivoices/submission_checkpoint.csv"
BATCH_SIZE      = 8
print(f"Test data at: {test_path}")

if os.path.exists(CHECKPOINT_FILE):
    existing  = pd.read_csv(CHECKPOINT_FILE)
    empty_pct = (existing["transcription"].isna() |
                 (existing["transcription"].str.strip() == "")).mean()
    if empty_pct > 0.5:
        os.remove(CHECKPOINT_FILE)
        print(f"Deleted corrupted checkpoint ({empty_pct:.0%} empty). Starting fresh.")
        results, done_ids = [], set()
    else:
        results  = existing.to_dict("records")
        done_ids = set(existing["id"])
        print(f"Resuming: {len(results)} clips done.")
else:
    results, done_ids = [], set()
    print("Starting fresh.")

all_parquet_files = sorted(
    glob.glob(os.path.join(test_path, "**", "*.parquet"), recursive=True)
)
print(f"{len(all_parquet_files)} parquet files. Batch size: {BATCH_SIZE}.\n")

for pf_idx, pq_file in enumerate(all_parquet_files):
    df = pd.read_parquet(pq_file)
    df = df[~df["id"].isin(done_ids)]
    if len(df) == 0:
        print(f"[{pf_idx+1}/{len(all_parquet_files)}] already done — skip")
        continue

    lang3   = df["language"].iloc[0]
    wh_lang = LANG_TO_WHISPER.get(lang3)
    print(f"[{pf_idx+1}/{len(all_parquet_files)}] {os.path.basename(pq_file)} "
          f"lang={lang3} rows={len(df)}", flush=True)

    rows = list(df.itertuples(index=False))
    for i in range(0, len(rows), BATCH_SIZE):
        chunk = rows[i: i + BATCH_SIZE]
        arrays, batch_ids, batch_langs = [], [], []
        for row in chunk:
            try:
                arrays.append(decode_audio(row.audio))
                batch_ids.append(row.id)
                batch_langs.append(row.language)
            except Exception as e:
                print(f"  DECODE ERROR {row.id}: {e}")
                results.append({"id": row.id, "language": row.language,
                                "transcription": "."})
                done_ids.add(row.id)

        if arrays:
            try:
                texts = transcribe_batch(arrays, language=wh_lang)
                for id_, lang_, text in zip(batch_ids, batch_langs, texts):
                    results.append({"id": id_, "language": lang_,
                                    "transcription": text.strip() or "."})
                    done_ids.add(id_)
            except Exception as e:
                print(f"  BATCH ERROR ({e}) — one-by-one")
                for id_, lang_, arr in zip(batch_ids, batch_langs, arrays):
                    try:
                        text = transcribe_batch([arr], language=wh_lang)[0].strip() or "."
                    except Exception as e2:
                        print(f"    FAILED {id_}: {e2}")
                        text = "."
                    results.append({"id": id_, "language": lang_,
                                    "transcription": text})
                    done_ids.add(id_)

    pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
    print(f"  → checkpoint saved ({len(results)} total)", flush=True)

print(f"\nDone. Total: {len(results)}")

In [ ]:
# ── CELL 13 — Build submission file ──────────────────────────────────────────
SUBMISSION_PATH = "/content/drive/MyDrive/afrivoices/submission.csv"

if not results and os.path.exists(CHECKPOINT_FILE):
    submission_df = pd.read_csv(CHECKPOINT_FILE)
else:
    submission_df = pd.DataFrame(results)

mask = (submission_df["transcription"].isna() |
        (submission_df["transcription"].str.strip() == ""))
if mask.sum() > 0:
    print(f"Replacing {mask.sum()} empty rows with '.'")
    submission_df.loc[mask, "transcription"] = "."

submission_df = submission_df[["id", "language", "transcription"]]
submission_df.to_csv(SUBMISSION_PATH, index=False)

sub = pd.read_csv(SUBMISSION_PATH)
print(f"Rows            : {len(sub)}")
print(f"NaN transcription: {sub['transcription'].isna().sum()}")
print(f"Empty            : {(sub['transcription'].str.strip() == '').sum()}")
print(sub.head())
print(f"\nsubmission.csv saved to Google Drive: {SUBMISSION_PATH}")
print("Download it and upload to the competition page → Submit Prediction")